# 11 - Multitask Segmentation + EF Fine-Tuning

This Kaggle-compatible notebook adds an EF regression head to the already-trained 23-frame bidirectional ConvLSTM U-Net and jointly fine-tunes segmentation plus EF prediction. The EF head uses only the fused bidirectional temporal bottleneck representation: no target-frame skip connections and no decoder features are exposed to the EF head.

The goal is to test whether EF supervision pushes the shared temporal representation toward true frame-to-frame cardiac dynamics rather than repeated target-frame appearance.


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import random
import sys
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def first_existing_path(candidates):
    cleaned = [candidate for candidate in candidates if candidate]
    for candidate in cleaned:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(cleaned[-1])


PROJECT_ROOT = first_existing_path([
    os.environ.get("PROJECT_ROOT"),
    "/kaggle/input/echonet-temporal-xai",
    "/kaggle/working/Echonet_temporal_XAI",
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
])
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.bidirectional_convlstm_unet import BidirectionalConvLSTMUNet
from src.dataset import EchoNetTemporalDataset, load_temporal_metadata, split_by_echonet_filelist
from src.temporal_train import get_temporal_loss, segmentation_metrics
from src.utils import load_echonet_tables, set_seed

RAW_DIR = Path(os.environ.get("ECHONET_RAW_DIR", PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"))
PROCESSED_DIR = Path(os.environ.get("ECHONET_PROCESSED_DIR", PROJECT_ROOT / "data" / "processed"))
VIDEOS_DIR = RAW_DIR / "Videos"
PRETRAINED_CHECKPOINT_DIR = Path(os.environ.get(
    "BIDIRECTIONAL_CONVLSTM_CHECKPOINT_DIR",
    PROJECT_ROOT / "outputs" / "runs" / "bidirectional_convlstm_unet_23_frames" / "checkpoints",
))
RUN_DIR = Path("/kaggle/working/outputs/runs/multitask_segmentation_ef") if Path("/kaggle/working").exists() else PROJECT_ROOT / "outputs" / "runs" / "multitask_segmentation_ef"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
FIGURES_DIR = RUN_DIR / "figures"
MANIFEST_DIR = RUN_DIR / "manifests"
QUAL_DIR = FIGURES_DIR / "qualitative_examples"

for directory in [RUN_DIR, CHECKPOINT_DIR, FIGURES_DIR, MANIFEST_DIR, QUAL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw EchoNet directory: {RAW_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Pretrained checkpoint directory: {PRETRAINED_CHECKPOINT_DIR}")
print(f"Output directory: {RUN_DIR}")


## Configuration


In [ ]:
RUN_MODE = "smoke"  # change to "full" for the complete experiment
NUM_FRAMES_BEFORE = 11
NUM_FRAMES_AFTER = 11
TEMPORAL_STRIDE = 2
TARGET_IDX = NUM_FRAMES_BEFORE
SEQUENCE_LENGTH = NUM_FRAMES_BEFORE + 1 + NUM_FRAMES_AFTER
IMAGE_SIZE = (112, 112)
CHANNELS = (16, 32, 64, 128)
THRESHOLD = 0.5

SMOKE_CONFIG = {
    "run_mode": "smoke",
    "seed": 42,
    "epochs": 1,
    "batch_size": 2,
    "num_workers": 2,
    "max_train_samples": 32,
    "max_val_samples": 16,
    "max_test_samples": 16,
    "pretrained_lr": 3e-5,
    "ef_head_lr": 1e-4,
    "weight_decay": 1e-5,
    "lambda_ef": 0.1,
    "primary_checkpoint_metric": "val_dice",  # higher is better
    "secondary_checkpoint_metric": "val_ef_mae",  # lower is better
    "ef_hidden_dim": 128,
    "dropout": 0.1,
}

FULL_CONFIG = {
    "run_mode": "full",
    "seed": 42,
    "epochs": 30,
    "batch_size": 4,
    "num_workers": 2,
    "max_train_samples": None,
    "max_val_samples": None,
    "max_test_samples": None,
    "pretrained_lr": 3e-5,
    "ef_head_lr": 1e-4,
    "weight_decay": 1e-5,
    "lambda_ef": 0.1,
    "primary_checkpoint_metric": "val_dice",
    "secondary_checkpoint_metric": "val_ef_mae",
    "ef_hidden_dim": 128,
    "dropout": 0.1,
}

config = SMOKE_CONFIG if RUN_MODE == "smoke" else FULL_CONFIG
config.update({
    "num_frames_before": NUM_FRAMES_BEFORE,
    "num_frames_after": NUM_FRAMES_AFTER,
    "temporal_stride": TEMPORAL_STRIDE,
    "target_idx": TARGET_IDX,
    "sequence_length": SEQUENCE_LENGTH,
    "image_size": list(IMAGE_SIZE),
    "channels": list(CHANNELS),
    "threshold": THRESHOLD,
})
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)
config


## Load EchoNet Labels and Official Split

EF labels are read from `FileList.csv`. The notebook fails if an EF column is unavailable; it does not fabricate labels.


In [ ]:
metadata_path = PROCESSED_DIR / "metadata.csv"
assert metadata_path.exists(), f"Missing processed metadata: {metadata_path}"
assert (RAW_DIR / "FileList.csv").exists(), f"Missing FileList.csv under {RAW_DIR}"
assert VIDEOS_DIR.exists(), f"Missing Videos directory: {VIDEOS_DIR}"

samples = load_temporal_metadata(metadata_path)
file_list, _tracings = load_echonet_tables(RAW_DIR)
assert "EF" in file_list.columns, "FileList.csv must contain an EF column for multitask training."

# EchoNet FileName may be stored with or without .avi. Use video stem as the common key.
echo_table = file_list.copy()
echo_table["video_stem"] = echo_table["FileName"].astype(str).map(lambda x: Path(x).stem)
ef_lookup = dict(zip(echo_table["video_stem"], echo_table["EF"].astype(float)))

samples_with_ef = []
missing_ef = []
for sample in samples:
    item = dict(sample)
    video_stem = Path(str(item["video_id"])).stem
    if video_stem not in ef_lookup or pd.isna(ef_lookup[video_stem]):
        missing_ef.append(str(item["id"]))
        continue
    item["ef"] = float(ef_lookup[video_stem])
    samples_with_ef.append(item)
assert samples_with_ef, "No processed samples could be matched to EF labels."
if missing_ef:
    print(f"Skipped {len(missing_ef)} samples without EF labels.")

train_samples, val_samples, test_samples = split_by_echonet_filelist(samples_with_ef, file_list)
full_split_counts = {"train": len(train_samples), "validation": len(val_samples), "test": len(test_samples)}
assert min(full_split_counts.values()) > 0, f"Empty split after EF merge: {full_split_counts}"

if config["max_train_samples"] is not None:
    train_samples = train_samples[:config["max_train_samples"]]
if config["max_val_samples"] is not None:
    val_samples = val_samples[:config["max_val_samples"]]
if config["max_test_samples"] is not None:
    test_samples = test_samples[:config["max_test_samples"]]

print(f"Processed samples with EF: {len(samples_with_ef):,}")
print(f"Official full split counts: {full_split_counts}")
print(f"Active train/val/test: {len(train_samples):,} / {len(val_samples):,} / {len(test_samples):,}")
print(file_list[["FileName", "EF"]].head())


## ED-to-ES Distance and 23-Frame Coverage Check

This cell uses explicit EchoNet ED/ES frame-index fields when they are present in the dataset tables. If those fields are unavailable, it falls back to the processed annotation frame pair and records that the ED/ES labels were inferred from annotation ordering. It reports two related but different quantities: exact sampled ED/ES coverage means both ED and ES are among the 23 sampled frames; ED/ES window coverage means both ED and ES fall within the temporal span covered by the sequence, even if stride 2 did not sample both frames exactly.


In [ ]:
def sequence_frame_indices_for_target(target_frame_idx: int, frame_count: int | None = None) -> list[int]:
    indices = [target_frame_idx + offset * TEMPORAL_STRIDE for offset in range(-NUM_FRAMES_BEFORE, NUM_FRAMES_AFTER + 1)]
    if frame_count is not None:
        indices = [min(max(idx, 0), frame_count - 1) for idx in indices]
    return indices


def find_explicit_ed_es_columns(file_list_df: pd.DataFrame) -> tuple[str | None, str | None]:
    # EchoNet releases may name these fields differently. Only use columns that explicitly encode frame indices.
    ed_candidates = ["EDFrame", "FrameED", "ED_Frame", "ED_frame", "EDFrameIndex", "ED_frame_idx", "ed_frame", "ed_frame_idx"]
    es_candidates = ["ESFrame", "FrameES", "ES_Frame", "ES_frame", "ESFrameIndex", "ES_frame_idx", "es_frame", "es_frame_idx"]
    lower_to_actual = {str(col).lower(): col for col in file_list_df.columns}
    ed_col = next((lower_to_actual[name.lower()] for name in ed_candidates if name.lower() in lower_to_actual), None)
    es_col = next((lower_to_actual[name.lower()] for name in es_candidates if name.lower() in lower_to_actual), None)
    return ed_col, es_col


def build_ed_es_lookup(samples_with_ef: list[dict[str, Any]], file_list_df: pd.DataFrame) -> tuple[dict[str, dict[str, Any]], str]:
    ed_col, es_col = find_explicit_ed_es_columns(file_list_df)
    filename_col = next((col for col in ["FileName", "filename", "video_id"] if col in file_list_df.columns), None)
    if ed_col is not None and es_col is not None and filename_col is not None:
        lookup = {}
        for _, row in file_list_df.iterrows():
            video_id = str(row[filename_col]).replace(".avi", "")
            if pd.notna(row[ed_col]) and pd.notna(row[es_col]):
                lookup[video_id] = {
                    "ed_frame_idx": int(row[ed_col]),
                    "es_frame_idx": int(row[es_col]),
                    "phase_source": f"explicit_filelist_columns:{ed_col},{es_col}",
                }
        return lookup, "explicit_dataset_fields"

    # Fallback: if explicit ED/ES frame-index columns are unavailable, infer the two annotated phases
    # from the processed segmentation targets. This assumes the lower/higher annotated frame pair
    # corresponds to the two EchoNet ED/ES tracing frames, but it should not be treated as an explicit label.
    annotated_by_video: dict[str, list[int]] = {}
    for sample in samples_with_ef:
        annotated_by_video.setdefault(str(sample["video_id"]), []).append(int(sample["frame_idx"]))
    lookup = {}
    for video_id, frames in annotated_by_video.items():
        unique_frames = sorted(set(int(frame) for frame in frames))
        if len(unique_frames) >= 2:
            lookup[video_id] = {
                "ed_frame_idx": unique_frames[0],
                "es_frame_idx": unique_frames[-1],
                "phase_source": "fallback_inferred_from_processed_annotation_order",
            }
    return lookup, "fallback_inferred_from_processed_annotation_order"


ed_es_lookup, ed_es_phase_source = build_ed_es_lookup(samples_with_ef, file_list)
coverage_rows = []
phase_lookup = {}
for sample in samples_with_ef:
    sample_id = str(sample["id"])
    video_id = str(sample["video_id"])
    target_frame = int(sample["frame_idx"])
    sampled = sequence_frame_indices_for_target(target_frame)
    info = ed_es_lookup.get(video_id, {})
    ed_frame = info.get("ed_frame_idx")
    es_frame = info.get("es_frame_idx")
    source = info.get("phase_source", "unavailable")
    if ed_frame is not None and es_frame is not None:
        distance = abs(int(es_frame) - int(ed_frame))
        contains_both_exact = int(ed_frame) in sampled and int(es_frame) in sampled
        contains_both_range = min(sampled) <= int(ed_frame) <= max(sampled) and min(sampled) <= int(es_frame) <= max(sampled)
        if target_frame == int(ed_frame):
            phase = "ED"
        elif target_frame == int(es_frame):
            phase = "ES"
        else:
            phase = "other_annotated"
    else:
        distance = None
        contains_both_exact = False
        contains_both_range = False
        phase = "unknown"
    phase_lookup[sample_id] = phase
    coverage_rows.append({
        "sample_id": sample_id,
        "video_id": video_id,
        "phase": phase,
        "phase_source": source,
        "target_frame_idx": target_frame,
        "ed_frame_idx": ed_frame,
        "es_frame_idx": es_frame,
        "ed_es_distance_frames": distance,
        "contains_both_annotated_frames_exact_sampled": bool(contains_both_exact),
        "ed_es_span_within_temporal_window": bool(contains_both_range),
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(MANIFEST_DIR / "ed_es_sequence_coverage.csv", index=False)
valid_distance = coverage_df["ed_es_distance_frames"].dropna()
coverage_summary = {
    "sample_count": int(len(coverage_df)),
    "ed_es_phase_source": ed_es_phase_source,
    "sequences_with_exact_sampled_ed_es_count": int(coverage_df["contains_both_annotated_frames_exact_sampled"].sum()),
    "percentage_exact_sampled_ed_es_coverage": float(100.0 * coverage_df["contains_both_annotated_frames_exact_sampled"].mean()),
    "sequences_with_ed_es_span_inside_temporal_window_count": int(coverage_df["ed_es_span_within_temporal_window"].sum()),
    "percentage_ed_es_span_inside_temporal_window": float(100.0 * coverage_df["ed_es_span_within_temporal_window"].mean()),
    "ed_es_distance_min_frames": float(valid_distance.min()) if len(valid_distance) else float("nan"),
    "ed_es_distance_median_frames": float(valid_distance.median()) if len(valid_distance) else float("nan"),
    "ed_es_distance_mean_frames": float(valid_distance.mean()) if len(valid_distance) else float("nan"),
    "ed_es_distance_max_frames": float(valid_distance.max()) if len(valid_distance) else float("nan"),
}
with (MANIFEST_DIR / "ed_es_sequence_coverage_summary.json").open("w", encoding="utf-8") as file:
    json.dump(coverage_summary, file, indent=2)
print(json.dumps(coverage_summary, indent=2))
print("Exact sampled ED/ES coverage: ED and ES are both included among the 23 sampled frames.")
print("ED/ES temporal-window coverage: ED and ES fall inside the covered frame range; stride 2 may skip one or both exact annotated frames.")
display(coverage_df["ed_es_distance_frames"].describe())



## Multitask Dataset Wrapper


In [ ]:
class EchoNetTemporalEFDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset: EchoNetTemporalDataset, ef_mean: float, ef_std: float) -> None:
        self.base_dataset = base_dataset
        self.ef_mean = float(ef_mean)
        self.ef_std = float(ef_std)

    def __len__(self) -> int:
        return len(self.base_dataset)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        item = dict(self.base_dataset[idx])
        ef = float(self.base_dataset.samples[idx]["ef"])
        item["ef"] = torch.tensor(ef, dtype=torch.float32)
        item["ef_normalized"] = torch.tensor((ef - self.ef_mean) / self.ef_std, dtype=torch.float32)
        item["phase"] = phase_lookup.get(str(item["id"]), "unknown")
        return item


ef_values_train = np.array([float(sample["ef"]) for sample in train_samples], dtype=np.float32)
ef_mean = float(ef_values_train.mean())
ef_std = float(ef_values_train.std(ddof=0))
assert ef_std > 0, "Training EF standard deviation is zero; cannot normalize EF."
print(f"EF normalization from training split: mean={ef_mean:.3f}, std={ef_std:.3f}")


## DataLoaders


In [ ]:
dataset_kwargs = {
    "videos_dir": VIDEOS_DIR,
    "num_frames_before": config["num_frames_before"],
    "num_frames_after": config["num_frames_after"],
    "temporal_stride": config["temporal_stride"],
    "image_size": tuple(config["image_size"]),
}
train_base = EchoNetTemporalDataset(train_samples, augment=True, **dataset_kwargs)
val_base = EchoNetTemporalDataset(val_samples, augment=False, **dataset_kwargs)
test_base = EchoNetTemporalDataset(test_samples, augment=False, **dataset_kwargs)
train_dataset = EchoNetTemporalEFDataset(train_base, ef_mean, ef_std)
val_dataset = EchoNetTemporalEFDataset(val_base, ef_mean, ef_std)
test_dataset = EchoNetTemporalEFDataset(test_base, ef_mean, ef_std)

loader_kwargs = {
    "batch_size": config["batch_size"],
    "num_workers": config["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": config["num_workers"] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

sample = train_dataset[0]
assert sample["sequence"].shape == (SEQUENCE_LENGTH, 1, *IMAGE_SIZE)
assert sample["mask"].shape == (1, *IMAGE_SIZE)
assert torch.isfinite(sample["ef_normalized"])
print(f"Sample sequence shape: {tuple(sample['sequence'].shape)}")
print(f"Sample EF: {float(sample['ef']):.2f}% | normalized={float(sample['ef_normalized']):.3f}")


## Multitask Model

The segmentation path matches the trained bidirectional ConvLSTM U-Net. The EF head receives only the target-aligned fused bidirectional temporal representation after `bidirectional_fusion`, followed by adaptive average pooling and a small MLP.


In [ ]:
class MultiTaskBidirectionalConvLSTMUNet(BidirectionalConvLSTMUNet):
    def __init__(self, *args, ef_hidden_dim: int = 128, dropout: float = 0.1, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        bottleneck_channels = self.bidirectional_fusion[0].out_channels
        self.ef_pool = nn.AdaptiveAvgPool2d(1)
        self.ef_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(bottleneck_channels, ef_hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(ef_hidden_dim, 1),
        )

    def forward(self, sequence: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        self._validate_sequence(sequence)
        bottlenecks, target_skips = self._encode_sequence(sequence)
        forward_hidden = self._run_temporal_branch(
            self.forward_temporal_bottleneck,
            bottlenecks,
            range(0, self.target_idx + 1),
        )
        backward_hidden = self._run_temporal_branch(
            self.backward_temporal_bottleneck,
            bottlenecks,
            range(self.expected_sequence_length - 1, self.target_idx - 1, -1),
        )
        fused_temporal = self.bidirectional_fusion(torch.cat([forward_hidden, backward_hidden], dim=1))
        ef_normalized = self.ef_head(self.ef_pool(fused_temporal)).squeeze(1)

        skip1, skip2, skip3 = target_skips
        x = self.decoder3(fused_temporal, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)
        seg_logits = self.output(x)
        return seg_logits, ef_normalized


model = MultiTaskBidirectionalConvLSTMUNet(
    in_channels=1,
    out_channels=1,
    channels=tuple(config["channels"]),
    num_frames_before=config["num_frames_before"],
    num_frames_after=config["num_frames_after"],
    ef_hidden_dim=config["ef_hidden_dim"],
    dropout=config["dropout"],
).to(device)

sample_batch = next(iter(train_loader))
with torch.no_grad():
    seg_logits, ef_pred = model(sample_batch["sequence"].to(device))
assert seg_logits.shape == sample_batch["mask"].shape
assert ef_pred.shape == sample_batch["ef_normalized"].shape
print(f"Seg logits: {tuple(seg_logits.shape)} | EF normalized pred: {tuple(ef_pred.shape)}")


## Load Segmentation Checkpoint Into Multitask Model


In [ ]:
def select_best_checkpoint(checkpoint_dir: Path) -> Path:
    exact = checkpoint_dir / "best_model.pt"
    if exact.exists():
        return exact
    candidates = sorted(path for path in checkpoint_dir.iterdir() if path.suffix in {".pt", ".pth", ".ckpt"})
    best_like = [path for path in candidates if "best" in path.name.lower()]
    if len(best_like) == 1:
        return best_like[0]
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")


PRETRAINED_CHECKPOINT_PATH = select_best_checkpoint(PRETRAINED_CHECKPOINT_DIR)
checkpoint = torch.load(PRETRAINED_CHECKPOINT_PATH, map_location="cpu")
state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
if any(key.startswith("module.") for key in state_dict):
    state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
load_result = model.load_state_dict(state_dict, strict=False)
missing_keys = list(load_result.missing_keys)
unexpected_keys = list(load_result.unexpected_keys)
expected_missing = [key for key in model.state_dict() if key.startswith("ef_pool") or key.startswith("ef_head")]
extra_missing = sorted(set(missing_keys) - set(expected_missing))
assert not extra_missing, f"Unexpected missing pretrained keys: {extra_missing}"
assert not unexpected_keys, f"Unexpected checkpoint keys: {unexpected_keys}"
assert missing_keys, "Expected EF-head keys to be missing from segmentation checkpoint."

load_report = {
    "checkpoint_path": str(PRETRAINED_CHECKPOINT_PATH),
    "missing_keys": missing_keys,
    "unexpected_keys": unexpected_keys,
    "expected_missing_ef_keys": expected_missing,
}
with (MANIFEST_DIR / "checkpoint_load_report.json").open("w", encoding="utf-8") as file:
    json.dump(load_report, file, indent=2)
print(json.dumps(load_report, indent=2))


## Losses, Metrics, and Training Helpers


In [ ]:
seg_loss_fn = get_temporal_loss()
ef_loss_fn = nn.SmoothL1Loss()


def denormalize_ef(ef_normalized: torch.Tensor | np.ndarray) -> torch.Tensor | np.ndarray:
    return ef_normalized * ef_std + ef_mean


def ef_metrics(pred_percent: np.ndarray, target_percent: np.ndarray) -> dict[str, float]:
    pred_percent = np.asarray(pred_percent, dtype=np.float64)
    target_percent = np.asarray(target_percent, dtype=np.float64)
    errors = pred_percent - target_percent
    mae = float(np.mean(np.abs(errors))) if len(errors) else float("nan")
    rmse = float(np.sqrt(np.mean(errors ** 2))) if len(errors) else float("nan")
    if len(errors) > 1 and np.std(pred_percent) > 1e-8 and np.std(target_percent) > 1e-8:
        corr = float(np.corrcoef(pred_percent, target_percent)[0, 1])
    else:
        corr = float("nan")
    return {"ef_mae": mae, "ef_rmse": rmse, "ef_pearson": corr}


def batch_ids(batch: dict[str, Any]) -> list[str]:
    ids = batch.get("id")
    return [str(x) for x in ids]


def multitask_step(batch: dict[str, Any], train: bool = False) -> dict[str, Any]:
    sequences = batch["sequence"].to(device, non_blocking=True)
    masks = batch["mask"].to(device, non_blocking=True)
    ef_target_norm = batch["ef_normalized"].to(device, non_blocking=True)
    seg_logits, ef_pred_norm = model(sequences)
    seg_loss = seg_loss_fn(seg_logits, masks)
    ef_loss = ef_loss_fn(ef_pred_norm, ef_target_norm)
    weighted_ef_loss = config["lambda_ef"] * ef_loss
    total_loss = seg_loss + weighted_ef_loss
    dice, iou = segmentation_metrics(seg_logits, masks, threshold=THRESHOLD)
    ef_pred_pct = denormalize_ef(ef_pred_norm.detach().cpu()).numpy()
    ef_target_pct = batch["ef"].detach().cpu().numpy()
    return {
        "total_loss": total_loss,
        "seg_loss": seg_loss,
        "ef_loss": ef_loss,
        "weighted_ef_loss": weighted_ef_loss,
        "dice": dice.detach().cpu(),
        "iou": iou.detach().cpu(),
        "ef_pred_pct": ef_pred_pct,
        "ef_target_pct": ef_target_pct,
        "seg_logits": seg_logits,
    }


def aggregate_epoch(rows: list[dict[str, Any]], prefix: str) -> dict[str, float]:
    total_samples = sum(row["n"] for row in rows)
    dice_all = np.concatenate([row["dice"] for row in rows]) if rows else np.array([])
    iou_all = np.concatenate([row["iou"] for row in rows]) if rows else np.array([])
    ef_pred = np.concatenate([row["ef_pred_pct"] for row in rows]) if rows else np.array([])
    ef_true = np.concatenate([row["ef_target_pct"] for row in rows]) if rows else np.array([])
    metrics = {
        f"{prefix}_total_loss": sum(row["total_loss"] * row["n"] for row in rows) / max(total_samples, 1),
        f"{prefix}_seg_loss": sum(row["seg_loss"] * row["n"] for row in rows) / max(total_samples, 1),
        f"{prefix}_ef_loss": sum(row["ef_loss"] * row["n"] for row in rows) / max(total_samples, 1),
        f"{prefix}_weighted_ef_loss": sum(row["weighted_ef_loss"] * row["n"] for row in rows) / max(total_samples, 1),
        f"{prefix}_dice": float(np.mean(dice_all)) if len(dice_all) else float("nan"),
        f"{prefix}_iou": float(np.mean(iou_all)) if len(iou_all) else float("nan"),
    }
    metrics.update({f"{prefix}_{key}": value for key, value in ef_metrics(ef_pred, ef_true).items()})
    return metrics


def run_epoch(loader: DataLoader, train: bool) -> dict[str, float]:
    model.train(train)
    rows = []
    for batch in tqdm(loader, desc="train" if train else "eval", leave=False):
        if train:
            optimizer.zero_grad(set_to_none=True)
        out = multitask_step(batch, train=train)
        if train:
            out["total_loss"].backward()
            optimizer.step()
        n = int(batch["sequence"].shape[0])
        rows.append({
            "n": n,
            "total_loss": float(out["total_loss"].detach().cpu()),
            "seg_loss": float(out["seg_loss"].detach().cpu()),
            "ef_loss": float(out["ef_loss"].detach().cpu()),
            "weighted_ef_loss": float(out["weighted_ef_loss"].detach().cpu()),
            "dice": out["dice"].numpy(),
            "iou": out["iou"].numpy(),
            "ef_pred_pct": out["ef_pred_pct"],
            "ef_target_pct": out["ef_target_pct"],
        })
    return aggregate_epoch(rows, "train" if train else "val")


## Optimizer With Differential Learning Rates


In [ ]:
ef_head_param_ids = {id(param) for param in model.ef_head.parameters()}
pretrained_params = [param for param in model.parameters() if id(param) not in ef_head_param_ids and param.requires_grad]
ef_head_params = [param for param in model.ef_head.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(
    [
        {"params": pretrained_params, "lr": config["pretrained_lr"]},
        {"params": ef_head_params, "lr": config["ef_head_lr"]},
    ],
    weight_decay=config["weight_decay"],
)

def count_numel(params: list[torch.nn.Parameter]) -> int:
    return int(sum(param.numel() for param in params if param.requires_grad))

total_trainable_params = count_numel([param for param in model.parameters()])
pretrained_trainable_params = count_numel(pretrained_params)
ef_head_trainable_params = count_numel(ef_head_params)
assert total_trainable_params == pretrained_trainable_params + ef_head_trainable_params
print(f"Total trainable parameters: {total_trainable_params:,}")
print(f"Pretrained trainable parameters: {pretrained_trainable_params:,}")
print(f"EF-head trainable parameters: {ef_head_trainable_params:,}")
print(f"Learning rates: pretrained={config['pretrained_lr']:.1e}, ef_head={config['ef_head_lr']:.1e}")


## Joint Fine-Tuning


In [ ]:
history = []
best_val_dice = -float("inf")
best_val_ef_mae = float("inf")
best_val_dice_epoch = None
best_val_ef_mae_epoch = None

for epoch in range(1, config["epochs"] + 1):
    train_metrics = run_epoch(train_loader, train=True)
    with torch.no_grad():
        val_metrics = run_epoch(val_loader, train=False)
    row = {"epoch": epoch, **train_metrics, **val_metrics}
    history.append(row)
    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_DIR / "history.csv", index=False)

    checkpoint_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": row,
        "config": config,
        "ef_mean": ef_mean,
        "ef_std": ef_std,
    }

    current_val_dice = float(row["val_dice"])
    if current_val_dice > best_val_dice:
        best_val_dice = current_val_dice
        best_val_dice_epoch = epoch
        torch.save(checkpoint_payload, CHECKPOINT_DIR / "best_model.pt")
        torch.save(checkpoint_payload, CHECKPOINT_DIR / "best_val_dice_model.pt")

    current_val_ef_mae = float(row["val_ef_mae"])
    if current_val_ef_mae < best_val_ef_mae:
        best_val_ef_mae = current_val_ef_mae
        best_val_ef_mae_epoch = epoch
        torch.save(checkpoint_payload, CHECKPOINT_DIR / "best_ef_mae_model.pt")

    torch.save(checkpoint_payload, CHECKPOINT_DIR / "final_model.pt")
    checkpoint_selection = {
        "primary_checkpoint": "best_model.pt",
        "primary_metric": "val_dice",
        "primary_best_value": best_val_dice,
        "primary_best_epoch": best_val_dice_epoch,
        "secondary_checkpoint": "best_ef_mae_model.pt",
        "secondary_metric": "val_ef_mae",
        "secondary_best_value": best_val_ef_mae,
        "secondary_best_epoch": best_val_ef_mae_epoch,
    }
    with (MANIFEST_DIR / "checkpoint_selection.json").open("w", encoding="utf-8") as file:
        json.dump(checkpoint_selection, file, indent=2)

    print(
        f"Epoch {epoch:03d}/{config['epochs']:03d} | "
        f"train_total={row['train_total_loss']:.4f} val_total={row['val_total_loss']:.4f} | "
        f"val_dice={row['val_dice']:.4f} val_iou={row['val_iou']:.4f} | "
        f"val_EF_MAE={row['val_ef_mae']:.2f} val_EF_RMSE={row['val_ef_rmse']:.2f} "
        f"val_EF_r={row['val_ef_pearson']:.3f} | "
        f"best_dice={best_val_dice:.4f} best_EF_MAE={best_val_ef_mae:.2f}"
    )

display(pd.DataFrame(history).tail())


## Prediction Tables and Final Test Metrics


In [ ]:
best_checkpoint = torch.load(CHECKPOINT_DIR / "best_model.pt", map_location=device)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()


@torch.no_grad()
def predict_table(
    loader: DataLoader,
    split: str,
    sequence_transform=None,
    model_to_eval: nn.Module | None = None,
) -> tuple[pd.DataFrame, dict[str, float]]:
    active_model = model if model_to_eval is None else model_to_eval
    active_model.eval()
    rows = []
    metric_rows = []
    for batch in tqdm(loader, desc=f"predict {split}", leave=False):
        sequences = batch["sequence"].to(device, non_blocking=True)
        if sequence_transform is not None:
            sequences = sequence_transform(sequences)
        masks = batch["mask"].to(device, non_blocking=True)
        seg_logits, ef_pred_norm = active_model(sequences)
        seg_loss = seg_loss_fn(seg_logits, masks)
        ef_loss = ef_loss_fn(ef_pred_norm, batch["ef_normalized"].to(device, non_blocking=True))
        weighted_ef_loss = config["lambda_ef"] * ef_loss
        total_loss = seg_loss + weighted_ef_loss
        dice, iou = segmentation_metrics(seg_logits, masks, threshold=THRESHOLD)
        pred = (torch.sigmoid(seg_logits) >= THRESHOLD).float()
        ef_pred_pct = denormalize_ef(ef_pred_norm.detach().cpu()).numpy()
        ef_true_pct = batch["ef"].detach().cpu().numpy()
        ids = batch_ids(batch)
        video_ids = [str(x) for x in batch.get("video_id", [""] * len(ids))]
        frame_idx = batch["frame_idx"].detach().cpu().numpy()
        pred_area = pred.sum(dim=(1, 2, 3)).detach().cpu().numpy()
        gt_area = (masks > 0.5).float().sum(dim=(1, 2, 3)).detach().cpu().numpy()
        metric_rows.append({
            "n": int(sequences.shape[0]),
            "total_loss": float(total_loss.detach().cpu()),
            "seg_loss": float(seg_loss.detach().cpu()),
            "ef_loss": float(ef_loss.detach().cpu()),
            "weighted_ef_loss": float(weighted_ef_loss.detach().cpu()),
            "dice": dice.detach().cpu().numpy(),
            "iou": iou.detach().cpu().numpy(),
            "ef_pred_pct": ef_pred_pct,
            "ef_target_pct": ef_true_pct,
        })
        for i, sample_id in enumerate(ids):
            rows.append({
                "split": split,
                "sample_id": sample_id,
                "video_id": video_ids[i],
                "target_frame_idx": int(frame_idx[i]),
                "phase": phase_lookup.get(sample_id, "unknown"),
                "ef_true": float(ef_true_pct[i]),
                "ef_pred": float(ef_pred_pct[i]),
                "ef_error": float(ef_pred_pct[i] - ef_true_pct[i]),
                "dice": float(dice[i].detach().cpu()),
                "iou": float(iou[i].detach().cpu()),
                "predicted_foreground_area": int(pred_area[i]),
                "ground_truth_foreground_area": int(gt_area[i]),
            })
    metrics = aggregate_epoch(metric_rows, split)
    return pd.DataFrame(rows), metrics


val_predictions_df, val_metrics = predict_table(val_loader, "validation")
test_predictions_df, test_metrics = predict_table(test_loader, "test")
val_predictions_df.to_csv(MANIFEST_DIR / "validation_predictions.csv", index=False)
test_predictions_df.to_csv(MANIFEST_DIR / "test_predictions.csv", index=False)
with (RUN_DIR / "test_metrics.json").open("w", encoding="utf-8") as file:
    json.dump(test_metrics, file, indent=2)
print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)
display(test_predictions_df.head())


## Training Curves and EF Scatter Plot


In [ ]:
history_df = pd.read_csv(RUN_DIR / "history.csv")
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()
for ax, train_col, val_col, title in [
    (axes[0], "train_total_loss", "val_total_loss", "Total loss"),
    (axes[1], "train_seg_loss", "val_seg_loss", "Segmentation loss"),
    (axes[2], "train_ef_loss", "val_ef_loss", "EF Smooth L1 loss"),
    (axes[3], "train_dice", "val_dice", "Dice"),
    (axes[4], "train_iou", "val_iou", "IoU"),
    (axes[5], "train_ef_mae", "val_ef_mae", "EF MAE"),
]:
    ax.plot(history_df["epoch"], history_df[train_col], label="train")
    ax.plot(history_df["epoch"], history_df[val_col], label="validation")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test_predictions_df["ef_true"], test_predictions_df["ef_pred"], s=18, alpha=0.7)
lims = [
    min(test_predictions_df["ef_true"].min(), test_predictions_df["ef_pred"].min()),
    max(test_predictions_df["ef_true"].max(), test_predictions_df["ef_pred"].max()),
]
ax.plot(lims, lims, color="black", linewidth=1)
ax.set_xlabel("Ground-truth EF (%)")
ax.set_ylabel("Predicted EF (%)")
ax.set_title("Test EF prediction")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "test_ef_pred_vs_true.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved figures to {FIGURES_DIR}")


## Temporal Dependence Evaluation After Multitask Fine-Tuning

The official test set is evaluated under six input conditions: normal chronology, repeated target frame, random frame order, reversed sequence, center-frame-only, and context-only with the target frame obscured. This tests whether EF supervision makes the temporal representation more sensitive to real cardiac dynamics.


In [ ]:
ABLATION_SEED = 42


def normal_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return sequences


def repeat_target_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return sequences[:, TARGET_IDX:TARGET_IDX + 1].repeat(1, SEQUENCE_LENGTH, 1, 1, 1).contiguous()


def context_shuffled_target_fixed_sequence(sequences: torch.Tensor) -> torch.Tensor:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(ABLATION_SEED)
    context_indices = [idx for idx in range(SEQUENCE_LENGTH) if idx != TARGET_IDX]
    shuffled_context = torch.tensor(context_indices, device=sequences.device)[torch.randperm(len(context_indices), generator=generator).to(sequences.device)]
    order = torch.empty(SEQUENCE_LENGTH, dtype=torch.long, device=sequences.device)
    order[TARGET_IDX] = TARGET_IDX
    order[torch.tensor(context_indices, device=sequences.device)] = shuffled_context
    assert int(order[TARGET_IDX].item()) == TARGET_IDX
    return sequences[:, order].contiguous()


def fully_shuffled_sequence(sequences: torch.Tensor) -> torch.Tensor:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(ABLATION_SEED)
    order = torch.randperm(SEQUENCE_LENGTH, generator=generator).to(sequences.device)
    return sequences[:, order].contiguous()


def reversed_sequence(sequences: torch.Tensor) -> torch.Tensor:
    return torch.flip(sequences, dims=(1,)).contiguous()


def center_frame_only_sequence(sequences: torch.Tensor) -> torch.Tensor:
    out = torch.zeros_like(sequences)
    out[:, TARGET_IDX] = sequences[:, TARGET_IDX]
    return out


def context_only_zero_target_sequence(sequences: torch.Tensor) -> torch.Tensor:
    out = sequences.clone()
    out[:, TARGET_IDX] = 0.0
    return out


def forward_with_temporal_branch_ablation(
    model_to_eval: nn.Module,
    sequence: torch.Tensor,
    zero_forward: bool = False,
    zero_backward: bool = False,
) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
    model_to_eval._validate_sequence(sequence)
    bottlenecks, target_skips = model_to_eval._encode_sequence(sequence)
    forward_hidden = model_to_eval._run_temporal_branch(
        model_to_eval.forward_temporal_bottleneck,
        bottlenecks,
        range(0, model_to_eval.target_idx + 1),
    )
    backward_hidden = model_to_eval._run_temporal_branch(
        model_to_eval.backward_temporal_bottleneck,
        bottlenecks,
        range(model_to_eval.expected_sequence_length - 1, model_to_eval.target_idx - 1, -1),
    )
    if zero_forward:
        forward_hidden = torch.zeros_like(forward_hidden)
        assert torch.count_nonzero(forward_hidden).item() == 0
    if zero_backward:
        backward_hidden = torch.zeros_like(backward_hidden)
        assert torch.count_nonzero(backward_hidden).item() == 0
    fused_temporal = model_to_eval.bidirectional_fusion(torch.cat([forward_hidden, backward_hidden], dim=1))

    ef_normalized = None
    if hasattr(model_to_eval, "ef_head"):
        ef_normalized = model_to_eval.ef_head(model_to_eval.ef_pool(fused_temporal)).squeeze(1)

    skip1, skip2, skip3 = target_skips
    x = model_to_eval.decoder3(fused_temporal, skip3)
    x = model_to_eval.decoder2(x, skip2)
    x = model_to_eval.decoder1(x, skip1)
    seg_logits = model_to_eval.output(x)
    if ef_normalized is None:
        return seg_logits
    return seg_logits, ef_normalized


def load_original_segmentation_model() -> BidirectionalConvLSTMUNet:
    original_model = BidirectionalConvLSTMUNet(
        in_channels=1,
        out_channels=1,
        channels=tuple(config["channels"]),
        num_frames_before=config["num_frames_before"],
        num_frames_after=config["num_frames_after"],
    ).to(device)
    checkpoint = torch.load(PRETRAINED_CHECKPOINT_PATH, map_location="cpu")
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", checkpoint))
    if any(key.startswith("module.") for key in state_dict):
        state_dict = {key.removeprefix("module."): value for key, value in state_dict.items()}
    original_model.load_state_dict(state_dict, strict=True)
    original_model.eval()
    return original_model


@torch.no_grad()
def predict_condition_table(
    model_to_eval: nn.Module,
    split: str,
    condition_name: str,
    sequence_transform=None,
    zero_forward: bool = False,
    zero_backward: bool = False,
    has_ef_head: bool = True,
) -> tuple[pd.DataFrame, dict[str, float]]:
    model_to_eval.eval()
    rows = []
    metric_rows = []
    for batch in tqdm(test_loader, desc=f"{split} {condition_name}", leave=False):
        sequences = batch["sequence"].to(device, non_blocking=True)
        if sequence_transform is not None:
            sequences = sequence_transform(sequences)
        masks = batch["mask"].to(device, non_blocking=True)
        if zero_forward or zero_backward:
            output = forward_with_temporal_branch_ablation(model_to_eval, sequences, zero_forward=zero_forward, zero_backward=zero_backward)
        else:
            output = model_to_eval(sequences)
        if has_ef_head:
            seg_logits, ef_pred_norm = output
            ef_loss = ef_loss_fn(ef_pred_norm, batch["ef_normalized"].to(device, non_blocking=True))
            weighted_ef_loss = config["lambda_ef"] * ef_loss
            ef_pred_pct = denormalize_ef(ef_pred_norm.detach().cpu()).numpy()
        else:
            seg_logits = output
            ef_loss = torch.tensor(float("nan"), device=device)
            weighted_ef_loss = torch.tensor(float("nan"), device=device)
            ef_pred_pct = np.full(int(sequences.shape[0]), np.nan, dtype=np.float32)
        seg_loss = seg_loss_fn(seg_logits, masks)
        total_loss = seg_loss + weighted_ef_loss if has_ef_head else seg_loss
        dice, iou = segmentation_metrics(seg_logits, masks, threshold=THRESHOLD)
        pred = (torch.sigmoid(seg_logits) >= THRESHOLD).float()
        ef_true_pct = batch["ef"].detach().cpu().numpy()
        ids = batch_ids(batch)
        video_ids = [str(x) for x in batch.get("video_id", [""] * len(ids))]
        frame_idx = batch["frame_idx"].detach().cpu().numpy()
        pred_area = pred.sum(dim=(1, 2, 3)).detach().cpu().numpy()
        gt_area = (masks > 0.5).float().sum(dim=(1, 2, 3)).detach().cpu().numpy()
        metric_rows.append({
            "n": int(sequences.shape[0]),
            "total_loss": float(total_loss.detach().cpu()),
            "seg_loss": float(seg_loss.detach().cpu()),
            "ef_loss": float(ef_loss.detach().cpu()),
            "weighted_ef_loss": float(weighted_ef_loss.detach().cpu()),
            "dice": dice.detach().cpu().numpy(),
            "iou": iou.detach().cpu().numpy(),
            "ef_pred_pct": ef_pred_pct,
            "ef_target_pct": ef_true_pct,
        })
        for i, sample_id in enumerate(ids):
            rows.append({
                "split": split,
                "sample_id": sample_id,
                "video_id": video_ids[i],
                "target_frame_idx": int(frame_idx[i]),
                "phase": phase_lookup.get(sample_id, "unknown"),
                "ef_true": float(ef_true_pct[i]),
                "ef_pred": float(ef_pred_pct[i]) if has_ef_head else np.nan,
                "ef_error": float(ef_pred_pct[i] - ef_true_pct[i]) if has_ef_head else np.nan,
                "dice": float(dice[i].detach().cpu()),
                "iou": float(iou[i].detach().cpu()),
                "predicted_foreground_area": int(pred_area[i]),
                "ground_truth_foreground_area": int(gt_area[i]),
                "model_variant": split,
                "inference_condition": condition_name,
            })
    metrics = aggregate_epoch(metric_rows, split)
    if not has_ef_head:
        metrics.update({f"{split}_ef_mae": np.nan, f"{split}_ef_rmse": np.nan, f"{split}_ef_pearson": np.nan})
    return pd.DataFrame(rows), metrics


INFERENCE_CONDITIONS = [
    {"name": "normal_chronological", "transform": normal_sequence},
    {"name": "target_repeated_23x", "transform": repeat_target_sequence},
    {"name": "context_shuffled_target_fixed", "transform": context_shuffled_target_fixed_sequence},
    {"name": "fully_shuffled", "transform": fully_shuffled_sequence},
    {"name": "sequence_reversed", "transform": reversed_sequence},
    {"name": "center_frame_only", "transform": center_frame_only_sequence},
    {"name": "context_only_target_zeroed", "transform": context_only_zero_target_sequence},
    {"name": "forward_temporal_zeroed", "transform": normal_sequence, "zero_forward": True},
    {"name": "backward_temporal_zeroed", "transform": normal_sequence, "zero_backward": True},
]

original_segmentation_model = load_original_segmentation_model()
model_variants = [
    {"name": "original_segmentation_model", "model": original_segmentation_model, "has_ef_head": False},
    {"name": "multitask_finetuned_model", "model": model, "has_ef_head": True},
]

condition_rows = []
condition_prediction_tables = []
for variant in model_variants:
    normal_dice = None
    for condition in INFERENCE_CONDITIONS:
        condition_name = condition["name"]
        pred_df, metrics = predict_condition_table(
            variant["model"],
            split=variant["name"],
            condition_name=condition_name,
            sequence_transform=condition.get("transform"),
            zero_forward=condition.get("zero_forward", False),
            zero_backward=condition.get("zero_backward", False),
            has_ef_head=variant["has_ef_head"],
        )
        condition_prediction_tables.append(pred_df)
        prefix = variant["name"]
        row = {
            "model_variant": variant["name"],
            "inference_condition": condition_name,
            "dice": metrics[f"{prefix}_dice"],
            "iou": metrics[f"{prefix}_iou"],
            "seg_loss": metrics[f"{prefix}_seg_loss"],
            "ef_mae": metrics.get(f"{prefix}_ef_mae", np.nan),
            "ef_rmse": metrics.get(f"{prefix}_ef_rmse", np.nan),
            "ef_pearson": metrics.get(f"{prefix}_ef_pearson", np.nan),
        }
        condition_rows.append(row)

condition_metrics_df = pd.DataFrame(condition_rows)
normal_lookup = condition_metrics_df[condition_metrics_df["inference_condition"] == "normal_chronological"].set_index("model_variant")["dice"].to_dict()
condition_metrics_df["normal_dice"] = condition_metrics_df["model_variant"].map(normal_lookup)
condition_metrics_df["dice_degradation_vs_normal"] = condition_metrics_df["normal_dice"] - condition_metrics_df["dice"]
condition_metrics_df.to_csv(MANIFEST_DIR / "temporal_condition_metrics.csv", index=False)
condition_metrics_df.to_csv(MANIFEST_DIR / "model_temporal_condition_comparison.csv", index=False)

condition_predictions_df = pd.concat(condition_prediction_tables, ignore_index=True)
normal_preds = condition_predictions_df[condition_predictions_df["inference_condition"] == "normal_chronological"][["model_variant", "sample_id", "dice", "iou", "ef_pred"]].rename(columns={
    "dice": "normal_dice",
    "iou": "normal_iou",
    "ef_pred": "normal_ef_pred",
})
condition_predictions_df = condition_predictions_df.merge(normal_preds, on=["model_variant", "sample_id"], how="left")
condition_predictions_df["dice_degradation_vs_normal"] = condition_predictions_df["normal_dice"] - condition_predictions_df["dice"]
condition_predictions_df["delta_iou_vs_normal"] = condition_predictions_df["iou"] - condition_predictions_df["normal_iou"]
condition_predictions_df["delta_ef_pred_vs_normal"] = condition_predictions_df["ef_pred"] - condition_predictions_df["normal_ef_pred"]
condition_predictions_df.to_csv(MANIFEST_DIR / "temporal_condition_predictions.csv", index=False)

display(condition_metrics_df)
display(condition_predictions_df.head())



## Qualitative Examples


In [ ]:
@torch.no_grad()
def save_multitask_qualitative_examples(max_examples: int = 8) -> pd.DataFrame:
    rows = []
    saved = 0
    model.eval()
    for batch in test_loader:
        sequences = batch["sequence"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)
        ids = batch_ids(batch)
        predictions = {}
        ef_predictions = {}
        condition_names = []
        for condition in INFERENCE_CONDITIONS:
            condition_name = condition["name"]
            condition_names.append(condition_name)
            transform = condition.get("transform", normal_sequence)
            transformed = transform(sequences)
            if condition.get("zero_forward", False) or condition.get("zero_backward", False):
                seg_logits, ef_pred_norm = forward_with_temporal_branch_ablation(
                    model,
                    transformed,
                    zero_forward=condition.get("zero_forward", False),
                    zero_backward=condition.get("zero_backward", False),
                )
            else:
                seg_logits, ef_pred_norm = model(transformed)
            predictions[condition_name] = (torch.sigmoid(seg_logits) >= THRESHOLD).float().detach().cpu().numpy()
            ef_predictions[condition_name] = denormalize_ef(ef_pred_norm.detach().cpu()).numpy()
        for i, sample_id in enumerate(ids):
            panels = [
                ("target", sequences[i, TARGET_IDX, 0].detach().cpu().numpy()),
                ("ground truth", masks[i, 0].detach().cpu().numpy()),
            ]
            panels.extend((name, predictions[name][i, 0]) for name in condition_names)
            cols = 4
            rows_n = int(math.ceil(len(panels) / cols))
            fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.0, rows_n * 3.0), squeeze=False)
            for panel_idx, ax in enumerate(axes.flat):
                ax.axis("off")
                if panel_idx < len(panels):
                    title, image = panels[panel_idx]
                    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
                    if title in ef_predictions:
                        title = f"{title}\nEF={ef_predictions[title][i]:.1f}%"
                    ax.set_title(title, fontsize=8)
            fig.suptitle(f"{sample_id} | true EF={float(batch['ef'][i]):.1f}%", fontsize=11)
            fig.tight_layout(rect=(0, 0, 1, 0.96))
            out_path = QUAL_DIR / f"{sample_id}_multitask_conditions.png"
            fig.savefig(out_path, dpi=150, bbox_inches="tight")
            plt.close(fig)
            rows.append({"sample_id": sample_id, "figure_path": str(out_path.relative_to(RUN_DIR))})
            saved += 1
            if saved >= max_examples:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

qualitative_example_count = int(config.get("qualitative_example_count", 8))
qualitative_df = save_multitask_qualitative_examples(qualitative_example_count)
qualitative_df.to_csv(MANIFEST_DIR / "qualitative_examples.csv", index=False)
display(qualitative_df)



## Outputs

Important outputs are written to `RUN_DIR`: training history, validation/test prediction CSVs, final test metrics, training curves, EF scatter plot, temporal-condition metrics and predictions, and qualitative examples.


In [ ]:
required_outputs = [
    RUN_DIR / "history.csv",
    RUN_DIR / "test_metrics.json",
    CHECKPOINT_DIR / "best_model.pt",
    CHECKPOINT_DIR / "best_val_dice_model.pt",
    CHECKPOINT_DIR / "best_ef_mae_model.pt",
    CHECKPOINT_DIR / "final_model.pt",
    MANIFEST_DIR / "checkpoint_selection.json",
    MANIFEST_DIR / "validation_predictions.csv",
    MANIFEST_DIR / "test_predictions.csv",
    MANIFEST_DIR / "temporal_condition_metrics.csv",
    MANIFEST_DIR / "model_temporal_condition_comparison.csv",
    MANIFEST_DIR / "temporal_condition_predictions.csv",
    FIGURES_DIR / "training_curves.png",
    FIGURES_DIR / "test_ef_pred_vs_true.png",
]
missing = [str(path) for path in required_outputs if not path.exists()]
assert not missing, f"Missing expected outputs: {missing}"
print(f"All outputs saved under: {RUN_DIR}")
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(RUN_DIR))
